# CAOA on one scan: every stage, visualised

The full pipeline with the **release models** (CAPCM, SEM, OCAM and retrieval; `scripts/make_release.py`)
on a single ScanNet scan, using the same code as `scripts/align_scene.py` (`caoa.inference`):

1. **Input scan**
2. **Instance segmentation**: SphericalMask (or SoftGroup) predictions, confidence threshold, kept classes
3. **Completion** (CAPCM): each instance completed in the world frame
4. **Retrieval, stage 1**: the retrieval embedding against all 52,472 ShapeNetV2 models, top *k*
5. **Symmetry** (SEM) of the candidate CADs
6. **Alignment and re-ranking, stage 2**: the Siamese OCAM aligns the object to every candidate; the best fit is kept
7. **Full output scene**: every retrieved CAD placed in the scan
8. **Ground truth** (annotated val scenes): the result against the S2C-C annotation, per object
9. **Timing**

Needs a GPU, the extras `train` and `notebook` (`uv sync --extra train --extra notebook`), ShapeNetCore v2 and ScanNet
(paths from `configs/paths*.yaml`). Figures are interactive (plotly): drag to rotate, click legend entries to hide.
Everything is drawn in the scan's own coordinates (z up).

In [ ]:
SCENE = "scene0207_00"      # a ScanNet scan; with an S2C-C annotation the ground-truth section runs too
SOURCE = "sphericalmask"    # instance predictions: "sphericalmask" or "softgroup"
CONF = 0.3                  # instance confidence threshold (the evaluation's)
K = 10                      # retrieval candidates per object
SEARCH = "class"            # "class": synsets of the predicted class; "all": every ShapeNetV2 model
RELEASE = None              # release folder; default <caoa_data>/release_v1
OCAM, SEED = "final", 777   # alignment model in the release (ocam/<OCAM>/seed<SEED>)
OBJ = None                  # instance shown in detail in the retrieval / alignment sections (None: the most confident)

In [ ]:
import time

import numpy as np

from caoa import inference as inf
from caoa import viz
from caoa.data import scan2cad
from caoa.data.pairs import load_pairs
from caoa.data.shapenet import load_cad_mesh
from caoa.data.instances import pred_dir, sg_pred_path
from caoa.eval.scan2cad_benchmark import category_name
from caoa.geometry import transform_points
from caoa.paths import load_paths

paths = load_paths()
release = inf.Release(RELEASE or paths.cache.parent / "release_v1", ocam=OCAM, seed=SEED)
annos = scan2cad.load_annotations(paths.s2c_annotations)
annotated = SCENE in annos
trs = annos[SCENE]["trs"] if annotated else None  # the scan's Scan2CAD frame if annotated, else ScanNet z-up -> y-up
ply = paths.scannet_scans / SCENE / f"{SCENE}_vh_clean_2.ply"
preds_file = sg_pred_path(pred_dir(paths, SOURCE), SCENE)
timing = {}
print(f"release {release.root} ({release.manifest['date']}, commit {release.manifest['git_commit'][:7]})")
print(f"{SCENE}: S2C-C annotation {'yes' if annotated else 'no'}; instances from {SOURCE}")


def timed(name, fn, *args, **kw):
    t = time.perf_counter()
    out = fn(*args, **kw)
    inf._sync()
    timing[name] = time.perf_counter() - t
    return out

## 1. Input scan

In [ ]:
scan = timed("load scan", inf.load_scan, ply, trs)
W2S = viz.world_to_scan(scan.trs)  # Scan2CAD world -> scan coordinates, for drawing world-frame results
viz.figure([viz.points(scan.points, scan.colors, name="scan", size=1.2)],
           f"{SCENE}: {len(scan.points):,} vertices").show()

## 2. Instance segmentation

Kept: confidence ≥ `CONF` and a class a CAD can be (door, window, picture, curtain and shower curtain are dropped).
Each kept instance goes through the rest of the pipeline; grey points belong to no kept instance.

In [ ]:
inst = timed("instances", inf.load_instances, scan, preds_file, CONF)
covered = np.any(inst.masks, axis=0) if inst.masks else np.zeros(len(scan.points), bool)
traces = [viz.points(scan.points[~covered], uniform="lightgrey", name="not in a kept instance", size=1, max_points=30_000)]
for i, (m, mask) in enumerate(zip(inst.meta, inst.masks)):
    traces.append(viz.points(scan.points[mask], uniform=viz.color(i), name=f"{i}: {m['label']} ({m['conf']:.2f})", size=1.5, max_points=6_000))
viz.figure(traces, f"{len(inst.pairs)} kept instances of {len(inst.pairs) + len(inst.dropped)} predicted").show()

low = sum(c < CONF for _, c in inst.dropped)
print(f"dropped: {low} below conf {CONF}, {len(inst.dropped) - low} of an excluded class / empty")
for i, m in enumerate(inst.meta):
    print(f"  {i:2d}  {m['label']:15s} conf {m['conf']:.2f}  {m['n_points']:6d} points")

## 3. Completion (CAPCM)

The observed instance points (colour) and the CAPCM completion (dark), per instance. The completion runs on the
instance as it is in the scan (world frame): no ground truth reaches it.

In [ ]:
comps = timed("completion", inf.complete_instances, release, inst)
comp_scan = [transform_points(comps[(p.scene_id, p.model_index)].astype(np.float64), W2S) for p in inst.pairs]
obs_scan = [transform_points(p.points.astype(np.float64), W2S) for p in inst.pairs]
panels = [[viz.points(o, uniform=viz.color(i), size=1.5, max_points=3_000), viz.points(c, uniform="#333333", size=1, max_points=4_000, opacity=0.5)]
          for i, (o, c) in enumerate(zip(obs_scan, comp_scan))]
viz.grid(panels, cols=5, titles=[f"{i}: {m['label']}" for i, m in enumerate(inst.meta)],
         title="observed (colour) and completed (dark) points").show()

In [ ]:
traces = [viz.points(scan.points, scan.colors, name="scan", size=1, max_points=40_000, opacity=0.3)]
traces += [viz.points(c, uniform=viz.color(i), name=f"{i}: {inst.meta[i]['label']} completion", size=1.2, max_points=4_000)
           for i, c in enumerate(comp_scan)]
viz.figure(traces, "completions in the scene").show()

## 4. Retrieval, stage 1

Each object's query (completion + instance points) is embedded and compared with every ShapeNetV2 model (`SEARCH`:
the synsets of its predicted class, or all). The top `K` go to stage 2. Shown for one object (`OBJ`).

In [ ]:
cands = timed("retrieval", inf.retrieve, release, inst, comps, K, SEARCH)
if OBJ is None:
    OBJ = int(np.argmax([m["conf"] for m in inst.meta]))  # the most confident kept instance
print(f"{'#':>2} {'class':15s} top-{K} synsets (stage-1 similarity of the top candidate)")
for i, (m, row, sim) in enumerate(zip(inst.meta, cands.keys, cands.similarity)):
    syn = sorted({category_name(k.split('_')[0]) for k in row})
    print(f"{i:2d} {m['label']:15s} {', '.join(syn)}  ({sim[0]:.3f})")

In [ ]:
def cad_panel(key, M=None, mesh_color="#888888", extra=()):
    catid, id_cad = key.split("_", 1)
    return [viz.mesh(load_cad_mesh(paths.shapenet, catid, id_cad), M, mesh_color=mesh_color, max_faces=8_000), *extra]

viz.grid([cad_panel(k) for k in cands.keys[OBJ]], cols=5,
         titles=[f"#{r + 1} {category_name(k.split('_')[0])}: sim {s:.3f}" for r, (k, s) in enumerate(zip(cands.keys[OBJ], cands.similarity[OBJ]))],
         title=f"object {OBJ} ({inst.meta[OBJ]['label']}): stage-1 candidates").show()

## 5. Alignment and re-ranking, stage 2 (with SEM)

For every candidate, SEM embeds the CAD's symmetry and the Siamese OCAM predicts the object's 9-DoF box from the
completion, the CAD and that embedding. The CAD placed on its box is scored against the object (`sum` = Chamfer to the
completion + to the instance points; lower is better), and the best fit is kept.

In [ ]:
al = timed("alignment", inf.align_candidates, release, inst, comps, cands, paths)
objects = inf.assemble(scan, inst, cands, al)
print(f"{'#':>2} {'class':15s} {'chosen CAD':45s} {'stage-1 rank':>12s} {'fit':>6s} {'P(symmetric)':>12s}")
for i, o in enumerate(objects):
    print(f"{i:2d} {o['label']:15s} {category_name(o['catid_cad'])}/{o['id_cad'][:30]:33s} {o['rank_stage1'] + 1:>12d} "
          f"{o['fit_score']:6.3f} {o['symmetric_prob'] if o['symmetric_prob'] is not None else float('nan'):12.2f}")

In [ ]:
i = OBJ
order = np.argsort(al.scores["sum"][i])
panels, titles = [], []
for b in order:
    pose = inf.candidate_pose(scan, cands, al, i, b)
    key = cands.keys[i][b]
    best = b == order[0]
    panels.append(cad_panel(key, pose["cad_to_scan"], mesh_color=viz.color(i) if best else "#999999",
                            extra=[viz.points(obs_scan[i], uniform="#222222", size=1, max_points=2_500)]))
    titles.append(f"{'★ ' if best else ''}fit {al.scores['sum'][i, b]:.3f} (stage 1 #{b + 1}), P(sym) {al.sym_probs[key][1] if al.sym_probs else float('nan'):.2f}")
viz.grid(panels, cols=5, titles=titles, height_per_row=300,
         title=f"object {i} ({inst.meta[i]['label']}): each candidate aligned to the observed points (black), best fit first").show()

In [ ]:
o = objects[OBJ]
traces = [viz.points(obs_scan[OBJ], uniform="#222222", name="observed points", size=1.5),
          viz.points(comp_scan[OBJ], uniform="#aaaaaa", name="completion", size=1, opacity=0.5),
          viz.box(o["box"]["center"], o["box"]["rotation"], o["box"]["extents"], viz.color(OBJ), "predicted box")]
traces += cad_panel(f"{o['catid_cad']}_{o['id_cad']}", o["cad_to_scan"], mesh_color=viz.color(OBJ))
traces[-1].name, traces[-1].showlegend, traces[-1].opacity = "retrieved CAD, aligned", True, 0.6
viz.figure(traces, f"object {OBJ}: final alignment").show()

## 6. Full output scene

In [ ]:
traces = [viz.points(scan.points, scan.colors, name="scan", size=1, max_points=50_000, opacity=0.25)]
for i, o in enumerate(objects):
    t = cad_panel(f"{o['catid_cad']}_{o['id_cad']}", o["cad_to_scan"], mesh_color=viz.color(i))[0]
    t.name, t.showlegend = f"{i}: {o['label']} → {category_name(o['catid_cad'])}", True
    traces.append(t)
viz.figure(traces, f"{SCENE}: {len(objects)} CADs retrieved and aligned", height=750).show()

## 7. Ground truth (annotated scenes)

Each S2C-C object of the scene, matched to an instance by point IoU ≥ 0.5 (one-to-one), as in the evaluation. Objects
without a match are not failures; they are listed as unmatched. Per matched object: the box criteria (Scan2CAD's
20 cm / 20° / 20% on the predicted box, CAD-independent), the CAOA metric (the paper's) and Scan2CAD's criteria with the
retrieved CAD (−center convention; a CAD other than the GT model rarely passes them).

In [ ]:
if not annotated:
    print("no S2C-C annotation for this scene")
else:
    val_pairs = load_pairs(paths.cache, "val")
    run = {"scan": scan, "instances": inst, "alignment": al, "objects": objects}
    rows = inf.evaluate_scene(run, paths.scannet_scans, annos, val_pairs)
    matched = [r for r in rows if r["matched"]]
    print(f"{len(rows)} S2C-C objects, {len(matched)} matched by an instance\n")
    print(f"{'category':10s} {'inst':>4s} {'IoU':>5s} {'box':>4s} {'CAOA':>5s} {'S2C':>4s} {'err t (m)':>9s} {'r (°)':>6s} {'s (%)':>6s}  CAD")
    for r in rows:
        if not r["matched"]:
            print(f"{r['category']:10s}  unmatched")
            continue
        cad = "exact GT model" if r["exact_cad"] else ("GT model in top-k" if r["gt_in_topk"] else "other model")
        yn = lambda b: "✓" if b else "✗"
        print(f"{r['category']:10s} {r['instance']:4d} {r['iou']:5.2f} {yn(r['box_ok']):>4s} {yn(r['caoa']):>5s} {yn(r['s2c_criteria']):>4s} "
              f"{r['err_t_m']:9.3f} {r['err_r_deg']:6.1f} {r['err_s_pct']:6.1f}  {cad}")
    if matched:
        print("\nmatched objects passing: " + ", ".join(f"{k} {np.mean([r[k] for r in matched]) * 100:.0f}%" for k in ("box_ok", "caoa", "s2c_criteria")))

In [ ]:
if annotated:
    anno = annos[SCENE]
    traces = [viz.points(scan.points, scan.colors, name="scan", size=1, max_points=40_000, opacity=0.2)]
    for n, r in enumerate(rows):
        p = next(q for q in val_pairs if q.scene_id == SCENE and q.model_index == r["model_index"])
        M = W2S @ scan2cad.trs_matrix(anno["aligned_models"][p.model_index]["trs"])  # GT CAD -> scan
        gt = cad_panel(r["gt_cad"], M, mesh_color="#2ca02c")[0]
        gt.name, gt.opacity, gt.showlegend = f"GT {r['category']}" + ("" if r["matched"] else " (unmatched)"), 0.35, True
        traces.append(gt)
        if r["matched"]:
            o = objects[r["instance"]]
            ok = r["box_ok"]
            traces.append(viz.box(o["box"]["center"], o["box"]["rotation"], o["box"]["extents"], "#1f77b4" if ok else "#d62728",
                                  f"pred {r['category']} ({'pass' if ok else 'fail'}, box criteria)"))
    viz.figure(traces, "GT CADs (green) vs predicted boxes (blue: pass, red: fail)", height=750).show()

## 8. Timing

In [ ]:
total = sum(timing.values())
for k, v in timing.items():
    print(f"{k:12s} {v:7.2f} s")
print(f"{'total':12s} {total:7.2f} s for {len(inst.pairs)} objects (models are loaded inside each stage)")